In [1]:
import os
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
from openai import OpenAI

from src.config import GraphRAGConfig  
from src.model import GraphRAGTrainer
from src.retriever import GraphRAGRetriever
from src.utils import load_graphrag_artifacts, networkx_to_torch_sparse, prepare_training_data_from_memory
from src.evaluation import run_experiment_pass_simple, save_summary_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Utilizzando il device: {device}")

✅ Utilizzando il device: cuda


In [2]:
# Cella 2
# --- PARAMETRI GLOBALI ---
DATASET = "confact" 
EMBEDDER_NAME = "nomic_gemma12"
MODEL_NAME = "gemma3:12b"
LLM_BASE_URL = "http://localhost:11504/v1"

# --- PARAMETRI DI RETRIEVAL ---
TEMPERATURE = 0.0
TOP_K = 10
TOP_J = 3
THRESHOLD = 0.5
NUM_HOPS = 0
DECAY_FACTOR = 0.5
REG_WEIGHT = 0.5
EPOCHS = 10
BATCH_SIZE = 32

# Inizializzazione Client OpenAI locale
client = OpenAI(base_url=LLM_BASE_URL, api_key="not-needed")
print(f"🤖 LLM configurato: {MODEL_NAME} su {LLM_BASE_URL}")

# Inizializzazione Modello Embedding
print("📥 Caricamento modello di embedding...")
embedding_model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    cache_folder="./models",
    trust_remote_code=True
)

🤖 LLM configurato: gemma3:12b su http://localhost:11504/v1
📥 Caricamento modello di embedding...


<All keys matched successfully>


In [3]:
ARTIFACTS_PATH = f"../GraphRAG/{DATASET}/ragtest/output/"
CSV_PATH = f"../GraphRAG/{DATASET}/{DATASET}.csv"
MODELS_DIR = f"./saved_models/{DATASET}"
RESULTS_DIR = f"./results/{DATASET}/{EMBEDDER_NAME}_reg{REG_WEIGHT}"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. Caricamento Grafo
print("--- 1. Loading Graph ---")
G, communities, chunk_text_map = load_graphrag_artifacts(ARTIFACTS_PATH, community_level=None)

# 2. Embedding dei Nodi
print("\n--- 2. Calcolo Embeddings Nodi ---")
nodes_list = list(G.nodes())
node_id_to_idx = {node: i for i, node in enumerate(nodes_list)}
idx_to_node_id = {i: node for node, i in node_id_to_idx.items()}

node_texts = [f"{n}: {G.nodes[n].get('description', '')}" for n in nodes_list]
node_embeddings_tensor = embedding_model.encode(node_texts, convert_to_tensor=True, device=device)

# 3. Matrice di Adiacenza
print("\n--- 3. Creazione Matrice di Adiacenza ---")
adjacency_matrix = networkx_to_torch_sparse(G, node_id_to_idx, device=device)

--- 1. Loading Graph ---
Loading data from: ../GraphRAG/confact/ragtest/output/
Indexed 255 chunks (Text Units).

--- 2. Calcolo Embeddings Nodi ---

--- 3. Creazione Matrice di Adiacenza ---


In [4]:
print("--- 4. Setup model ---")
config = GraphRAGConfig(
    token_dim=node_embeddings_tensor.shape[1],
    node_dim=node_embeddings_tensor.shape[1],
    device=device,
    learning_rate=1e-3,
    graph_regularization_weight=REG_WEIGHT
)

trainer = GraphRAGTrainer(
    config=config,
    node_embeddings=node_embeddings_tensor,
    adjacency_matrix=adjacency_matrix
)

model_path = os.path.join(MODELS_DIR, f"graphrag_reg{REG_WEIGHT}_epochs{EPOCHS}_lr0.001.pt")

if os.path.exists(model_path):
    print(f"Pre-trained model found in: {model_path}")
    trainer.model.load_state_dict(torch.load(model_path, map_location=device))
    trainer.model.eval()
else:
    print(f"No model found in {model_path}. Starting training...")
    train_data = prepare_training_data_from_memory(G, node_id_to_idx, embedding_model, device=device)
    split = int(0.8 * len(train_data))
    
    trainer.train(
        train_data=train_data[:split],
        val_data=train_data[split:],
        num_epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )
    torch.save(trainer.model.state_dict(), model_path)
    print(f"Saved model in: {model_path}")

retriever = GraphRAGRetriever(
    model=trainer.model,
    node_embeddings=trainer.node_embeddings,
    adjacency_matrix=trainer.adjacency_matrix, 
    device=device
)

--- 4. Setup model ---
🚀 Improved Model initialized on cuda
   Nodes: 2185, Dim: 768
Pre-trained model found in: ./saved_models/confact/graphrag_reg0.5_epochs10_lr0.001.pt


In [6]:
# Cella 5
print("--- 5. Avvio Esperimenti (Evaluation) ---")

prompt_simple_baseline = """You are a strict fact-checking assistant. 
Based ONLY on the provided context, classify the claim into one of these categories:
1. supported (The context explicitly confirms the claim)
2. refuted (The context explicitly contradicts the claim)

Context:
{evidence}

Claim: {claim}

Instructions: Answer ONLY with the label (supported or refuted). Do not explain.

Final Answer:"""

# Caricamento Claim
df = pd.read_csv(CSV_PATH, quotechar='"', escapechar='\\', encoding="utf-8", engine="python", on_bad_lines='skip').fillna("")

experiment_name = "Simple_Baseline"
run_id = 1

# Esecuzione del processo di inferenza
run_data = run_experiment_pass_simple(
    df=df,
    retriever=retriever,
    model=embedding_model,
    G=G,
    idx_to_node_id=idx_to_node_id,
    chunk_text_map=chunk_text_map,
    client=client,
    MODEL_NAME=MODEL_NAME,
    embedder=EMBEDDER_NAME,
    prompt_template=prompt_simple_baseline,
    run_id=run_id,
    experiment_name=experiment_name,
    top_k=TOP_K,
    top_j=TOP_J,
    threshold=THRESHOLD,
    num_hops=NUM_HOPS,
    decay_factor=DECAY_FACTOR
)

# Salvataggio e metriche
final_df = pd.DataFrame(run_data)
raw_csv_path = os.path.join(RESULTS_DIR, f"{experiment_name}_run{run_id}_raw.csv")
final_df.to_csv(raw_csv_path, index=False)
print(f"💾 Dati raw salvati in {raw_csv_path}")

summary_filename = os.path.join(RESULTS_DIR, f"summary_metrics_threshold{THRESHOLD}_topj{TOP_J}.csv")
summary_df = save_summary_metrics(final_df, summary_filename=summary_filename)

print("\n🏆 Anteprima Metriche:")
print(summary_df[["experiment_name", "run_id", "accuracy", "f1_macro"]].head())

--- 5. Avvio Esperimenti (Evaluation) ---
🔄 Run #1 [Simple_Baseline] - Elaborazione 2 claims (Simple Retrieval)...
🔍 Nodi trovati: 2. Chunk candidati unici: 22
Claim 0: Nigeria has an estimated physician-patient ratio of one doctor to every 4,000 to 5,000 patients.
Full Answer:  supported
prediction:  supported
GT:  supported
🔍 Nodi trovati: 2. Chunk candidati unici: 22
Claim 1: Nigeria has an estimated physician-patient ratio of one doctor to every 4,000 to 5,000 patients.
Full Answer:  supported
prediction:  supported
GT:  supported
💾 Dati raw salvati in ./results/confact/nomic_gemma12_reg0.5/Simple_Baseline_run1_raw.csv

📊 Generazione riepilogo in './results/confact/nomic_gemma12_reg0.5/summary_metrics_threshold0.5_topj3.csv'...
✅ File di riepilogo salvato con successo!

🏆 Anteprima Metriche:
   experiment_name  run_id  accuracy  f1_macro
0  Simple_Baseline       1       1.0       0.5
